In [1]:
# Baixando o groq que nos permite usar alguns modelos de LLM
%pip install groq
%pip install python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 2.9 MB/s eta 0:00:00


In [13]:
import os
from dotenv import load_dotenv

#Obter chave da API do groq
load_dotenv()
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [14]:
import os
from groq import Groq
# Configurando o modelo que usaremos para esse teste
client = Groq(
    api_key=os.environ.get("GROQ_API_KEY"),
)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Explain the importance of fast language models",
        }
    ],
    model="llama-3.3-70b-versatile",
)

print(chat_completion.choices[0].message.content)

Fast language models are crucial in today's technology landscape, and their importance can be understood from several perspectives:

1. **Efficient Processing**: Fast language models can process and analyze vast amounts of text data quickly, which is essential for many applications such as:
	* Sentiment analysis
	* Text classification
	* Named Entity Recognition (NER)
	* Machine Translation
	* Summarization
2. **Real-time Applications**: Fast language models enable real-time applications, such as:
	* Chatbots and virtual assistants
	* Language translation apps
	* Sentiment analysis for social media monitoring
	* Live text analysis for news and media outlets
3. **Improved User Experience**: Fast language models provide a seamless and responsive user experience, which is critical for:
	* Voice assistants (e.g., Alexa, Google Assistant)
	* Language-based interfaces (e.g., Siri, Cortana)
	* Interactive systems (e.g., customer service chatbots)
4. **Cost-Effectiveness**: Fast language model

Criando a classe Agent

In [15]:
class Agent: # Inicializamos o Agent com um cliente e um sistema
  def __init__(self, client: Groq, system: str = "") -> None:
      self.client = client
      self.system = system # Essa é a mensagem do sistema, isto é o system prompt (basicamente um roteiro de trabalho para o Agente)
      self.messages: list = []
      if self.system is not None:
          self.messages.append({"role": "system", "content": system}) # Desde que haja a mensagem do sistema, será incluida na lista de mensagens, a qual fornece as instruções iniciais para o modelo

  def __call__(self, message=""): # É acionada sempre que é feita uma chamada ao Agente
      if message:
          self.messages.append({"role": "user", "content": message})
      result = self.execute()
      self.messages.append({"role": "assistant", "content": result})
      return result

  def execute(self): # Executa uma conclusão com todo histórico de mensagem
      completion = client.chat.completions.create(
          messages=self.messages,
          model="llama-3.3-70b-versatile",
      )
      return completion.choices[0].message.content

Criando o System Prompt

In [16]:
# No System Prompt é passado uma espécie de roteiro para o Agente seguir
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

list_people:
e.g. list_people:
Returns the names available in the dataset.

get_person_data:
e.g. get_person_data: Ana
Returns name, weight, height, and age for that person.

calculate_imc:
e.g. calculate_imc: 70,1.75
Returns the BMI (IMC) rounded to 2 decimals.

classify_imc:
e.g. classify_imc: 22.86
Returns the BMI classification using the WHO table.

Example session:

Question: Calcule o IMC e a classificação de Ana?
Thought: Para calcular o IMC e a classificação de Ana, primeiro preciso saber o peso e a altura dele.
Action: get_person_data: Ana
PAUSE

You will be called again with this:

Observation: nome=Ana, peso=52.0, altura=1.6, idade=20

Thought: Agora que tenho o peso (52.0 kg) e a altura (1.6 m) de Ana, posso calcular o IMC usando a fórmula: IMC = peso / (altura^2). Após calcular o IMC, posso classificá-lo de acordo com a tabela da OMS.
Action: calculate_imc: 52.0, 1.6
PAUSE

You will be called again with this:

Observation: 20.31

Thought: Agora que tenho o IMC (20.31) de Ana, posso classificá-lo de acordo com a tabela da OMS. A classificação do IMC é importante para entender o estado de nutrição de uma pessoa.
Action: classify_imc: 20.31
PAUSE

If you have the answer, output it as the Answer.

Answer: O IMC de Ana é 20.31, o que é classificado como "Peso normal" de acordo com a tabela da OMS.

Now it's your turn:
""".strip()


Escrevendo a Lista de Pessoas disponíveis para o nosso Agente

In [17]:
PESSOAS = {
    "Ana":    {"peso": 52.0, "altura": 1.60, "idade": 20},
    "André":  {"peso": 85.0, "altura": 1.78, "idade": 24},
    "Maria":  {"peso": 68.0, "altura": 1.65, "idade": 32},
    "Rogério":  {"peso": 95.0, "altura": 1.72, "idade": 41},
    "Leonardo":    {"peso": 73.5, "altura": 1.80, "idade": 19},
    "Letícia":   {"peso": 59.0, "altura": 1.58, "idade": 27},
    "Thais":   {"peso": 110.0,"altura": 1.70, "idade": 36},
    "Victor":   {"peso": 77.0, "altura": 1.69, "idade": 29},
    "Fernanda":   {"peso": 44.0, "altura": 1.55, "idade": 22},
    "Lucas":   {"peso": 130.0,"altura": 1.75, "idade": 45},
}

Escrevendo as funções disponíveis para o Agente, isto é as ferramentas do Agente.

In [18]:
# As ferramentas (tools) disponíveis para um Agente são essencialmente funções
def list_people(_=""): # Retorna nomes disponíveis
    return ", ".join(PESSOAS.keys())

def get_person_data(name: str): # Obtém as informações da pessoa, caso ela esteja na lista de pessoas disponíveis
    name = name.strip()
    if name not in PESSOAS:
        return f"Pessoa '{name}' não encontrada. Use list_people."
    p = PESSOAS[name]
    return f"nome={name}, peso={p['peso']}, altura={p['altura']}, idade={p['idade']}"

def calculate_imc(args: str):
    """
    args no formato: "peso,altura"
    exemplo: "70,1.75"
    """
    parts = [x.strip() for x in args.split(",")]
    if len(parts) != 2:
        return "Formato inválido. Use: peso,altura (ex: 70,1.75)"
    peso = float(parts[0])
    altura = float(parts[1])
    imc = peso / (altura ** 2)
    return round(imc, 2)

def classify_imc(imc_value: str):
    """
    imc_value pode vir como "22.86" (string) e será convertido para float.
    Classificação OMS:
      <18.5: Abaixo do peso normal
      18.5-24.9: Peso normal
      25.0-29.9: Excesso de peso
      30.0-34.9: Obesidade classe I
      35.0-39.9: Obesidade classe II
      >=40.0: Obesidade classe III
    """
    imc = float(str(imc_value).strip())

    if imc < 18.5:
        return "Abaixo do peso normal"
    elif imc < 25.0:
        return "Peso normal"
    elif imc < 30.0:
        return "Excesso de peso"
    elif imc < 35.0:
        return "Obesidade classe I"
    elif imc < 40.0:
        return "Obesidade classe II"
    else:
        return "Obesidade classe III"

Inicializando nosso Agente (sem loop)

In [37]:
paulo_muzy = Agent(client, system_prompt)

In [38]:
result = paulo_muzy("Calcule o IMC e a classificação de Leonardo")
print(result)

Thought: Para calcular o IMC e a classificação de Leonardo, preciso primeiro saber o seu peso e altura. Após obter essas informações, posso calcular o IMC e, em seguida, classificá-lo de acordo com a tabela da OMS.

Action: get_person_data: Leonardo
PAUSE


In [39]:
paulo_muzy.messages # Essa linha permite a gente ver que primeiro o agente é inicalizado com a mensagem do sistema e depois que a do usuário

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\nlist_people:\ne.g. list_people:\nReturns the names available in the dataset.\n\nget_person_data:\ne.g. get_person_data: Ana\nReturns name, weight, height, and age for that person.\n\ncalculate_imc:\ne.g. calculate_imc: 70,1.75\nReturns the BMI (IMC) rounded to 2 decimals.\n\nclassify_imc:\ne.g. classify_imc: 22.86\nReturns the BMI classification using the WHO table.\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply th

In [40]:
observation = get_person_data("Leonardo")
print(observation)

nome=Leonardo, peso=73.5, altura=1.8, idade=19


In [41]:
next_prompt = f"Observation: {observation}"
result = paulo_muzy(next_prompt)
print(result)

Thought: Agora que tenho o peso (73.5 kg) e a altura (1.8 m) de Leonardo, posso calcular o IMC usando a fórmula: IMC = peso / (altura^2). Após calcular o IMC, posso classificá-lo de acordo com a tabela da OMS.

Action: calculate_imc: 73.5, 1.8
PAUSE


In [42]:
paulo_muzy.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\nlist_people:\ne.g. list_people:\nReturns the names available in the dataset.\n\nget_person_data:\ne.g. get_person_data: Ana\nReturns name, weight, height, and age for that person.\n\ncalculate_imc:\ne.g. calculate_imc: 70,1.75\nReturns the BMI (IMC) rounded to 2 decimals.\n\nclassify_imc:\ne.g. classify_imc: 22.86\nReturns the BMI classification using the WHO table.\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply th

In [44]:
observation = calculate_imc("73.5,1.8")
print(observation)

22.69


In [45]:
next_prompt = f"Observation: {observation}"
result = paulo_muzy(next_prompt)
print(result)

Thought: Agora que tenho o IMC (22.69) de Leonardo, posso classificá-lo de acordo com a tabela da OMS. A classificação do IMC é importante para entender o estado de nutrição de uma pessoa.

Action: classify_imc: 22.69
PAUSE


In [46]:
paulo_muzy.messages

[{'role': 'system',
  'content': "You run in a loop of Thought, Action, PAUSE, Observation.\nAt the end of the loop you output an Answer\nUse Thought to describe your thoughts about the question you have been asked.\nUse Action to run one of the actions available to you - then return PAUSE.\nObservation will be the result of running those actions.\n\nYour available actions are:\n\nlist_people:\ne.g. list_people:\nReturns the names available in the dataset.\n\nget_person_data:\ne.g. get_person_data: Ana\nReturns name, weight, height, and age for that person.\n\ncalculate_imc:\ne.g. calculate_imc: 70,1.75\nReturns the BMI (IMC) rounded to 2 decimals.\n\nclassify_imc:\ne.g. classify_imc: 22.86\nReturns the BMI classification using the WHO table.\n\nExample session:\n\nQuestion: What is the mass of Earth times 2?\nThought: I need to find the mass of Earth\nAction: get_planet_mass: Earth\nPAUSE\n\nYou will be called again with this:\n\nObservation: 5.972e24\n\nThought: I need to multiply th

In [47]:
observation = classify_imc("22.69")
print(observation)

Peso normal


In [48]:
next_prompt = f"Observation: {observation}"
result = paulo_muzy(next_prompt)
print(result)

Thought: Com o IMC calculado como 22.69 e a classificação como "Peso normal", tenho todas as informações necessárias para responder à pergunta. O IMC é uma medida importante para avaliar o estado de nutrição e saúde de uma pessoa.

Answer: O IMC de Leonardo é 22.69, o que é classificado como "Peso normal" de acordo com a tabela da OMS.


Rodando agora o Agente com loop

In [35]:
import re

def agent_loop(max_iterations, system, query):
  agent = Agent(client, system_prompt) # incializa o Agente
  tools = ['list_people', 'get_person_data', 'calculate_imc', 'classify_imc'] # inicializa os nomes das 'tools' que se tem disponível
  next_prompt = query # O primeiro prompt é definido como a pergunta que o user fizer para o Agente
  i = 0
  while i < max_iterations: # Inicializa-se o loop, com um máximo de interações que é definida na chamada da função
    i += 1
    result = agent(next_prompt) # Chama o Agente
    print(result)

    if "PAUSE" in result and "Action" in result: # Verifica o resultado obtido da chamada do Agente
      action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
      chosen_tool = action[0][0]
      arg = action[0][1]

      if chosen_tool in tools: # Verifica se a ferramenta escolhida está na lista de ferramentas disponíveis
        result_tool = eval(f"{chosen_tool}('{arg}')")
        next_prompt = f"Observation: {result_tool}"

      else:
        next_prompt = "Observation: Tool not found"

      print(next_prompt)
      continue # Serve para encerrar o primeiro 'if' do loop, para que reinicie o loop

    if "Answer" in result:
      break # Serve para avisar que terminou a tarefa e encerra o loop


In [36]:
agent_loop(max_iterations=10, system=system_prompt, query="Calcule o IMC e a classificação de: Ana, Victor, Fernanda. Mostre peso, altura, idade, IMC e classificação.")

Thought: Para calcular o IMC e a classificação de Ana, Victor e Fernanda, precisamos primeiro obter os dados de peso, altura e idade de cada uma dessas pessoas. Vamos começar com Ana.

Action: get_person_data: Ana
PAUSE
Observation: nome=Ana, peso=52.0, altura=1.6, idade=20
Thought: Agora que temos os dados de Ana, podemos calcular o seu IMC usando a fórmula IMC = peso / altura^2. Em seguida, classificaremos o IMC de acordo com a tabela da OMS.

Action: calculate_imc: 52.0, 1.6
PAUSE
Observation: 20.31
Thought: Com o IMC calculado, podemos agora classificar o IMC de Ana de acordo com a tabela da OMS. Além disso, precisamos repetir o processo para Victor e Fernanda.

Action: classify_imc: 20.31
PAUSE
Observation: Peso normal
Thought: Agora que temos os dados completos de Ana (nome, peso, altura, idade, IMC e classificação), podemos prosseguir com Victor. Precisamos obter os dados de Victor.

Action: get_person_data: Victor
PAUSE
Observation: nome=Victor, peso=77.0, altura=1.69, idade=29